In [3]:
import pandas as pd

# 원본 데이터 로드 (전국)
df = pd.read_csv(
    "/Users/mac/Desktop/project/company_data/third_week/data_csv/강원일반음식점.csv",
    encoding="CP949",
    low_memory=False
)

# 날짜 처리
df["인허가일자"] = pd.to_datetime(df["인허가일자"], errors="coerce")
df["폐업일자"] = pd.to_datetime(df["폐업일자"], errors="coerce")

# 최근 5년만 사용
df = df[df["인허가일자"] >= "2019-01-01"]

# 폐업여부 생성
df["폐업여부"] = df["폐업일자"].notnull().astype(int)

# 전국 → 강원도 필터링
df_gw = df[df["소재지전체주소"].str.contains("강원", na=False)].copy()

# 업종별 폐업률 계산
업종_폐업률 = (
    df_gw.groupby("위생업태명")
         .agg(
             총업소수=("위생업태명", "count"),
             폐업수=("폐업여부", "sum")
         )
)

업종_폐업률["폐업률"] = 업종_폐업률["폐업수"] / 업종_폐업률["총업소수"] * 100
업종_폐업률 = 업종_폐업률.sort_values("폐업률", ascending=False)

print(업종_폐업률)

                 총업소수   폐업수         폐업률
위생업태명                                  
까페                  1     1  100.000000
기타               6335  3068   48.429361
외국음식전문점(인도,태국등)   156    67   42.948718
키즈카페                7     3   42.857143
분식                857   365   42.590432
한식               7305  2978   40.766598
횟집                485   185   38.144330
정종/대포집/소주방        468   168   35.897436
패밀리레스트랑            32    11   34.375000
감성주점               24     8   33.333333
김밥(도시락)            48    16   33.333333
경양식               708   229   32.344633
호프/통닭            1135   364   32.070485
라이브카페              44    14   31.818182
뷔페식                83    26   31.325301
복어취급                7     2   28.571429
탕류(보신용)            39    10   25.641026
중국식               471   118   25.053079
일식                363    89   24.517906
냉면집                47    11   23.404255
식육(숯불구이)          849   198   23.321555
출장조리                6     1   16.666667
통닭(치킨)              2     0    0.000000


In [4]:
import pandas as pd

# 1) 데이터 로드
df = pd.read_csv(
    "/Users/mac/Desktop/project/company_data/third_week/data_csv/전국일반음식점.csv",
    encoding="CP949",
    low_memory=False
)

# 2) 날짜 처리
df["인허가일자"] = pd.to_datetime(df["인허가일자"], errors="coerce")
df["폐업일자"] = pd.to_datetime(df["폐업일자"], errors="coerce")

# 3) 강원도만 필터링
df = df[df["소재지전체주소"].astype(str).str.contains("강원")]

# 4) 분석 기간 2019~2023 필터링
df = df[df["인허가일자"] >= "2019-01-01"]

# 5) 폐업 여부
df["폐업여부"] = df["폐업일자"].notnull().astype(int)

# 6) 연도 컬럼
df["year"] = df["인허가일자"].dt.year

# 7) 연도·업종별 Net Growth 계산
annual = (
    df.groupby(["위생업태명", "year"])
      .agg(
          신규=("인허가일자", "count"),
          폐업=("폐업여부", "sum")
      )
      .reset_index()
)
annual["net_growth"] = annual["신규"] - annual["폐업"]

# 8) 강원도 전체 업종별 폐업률 계산
폐업률표 = (
    df.groupby("위생업태명")
      .agg(
          총업소수=("폐업여부", "count"),
          폐업수=("폐업여부", "sum")
      )
)
폐업률표["폐업률"] = 폐업률표["폐업수"] / 폐업률표["총업소수"] * 100

# 9) 최종 병합: 업종별 전체 Net Growth(5년 합계) + 폐업률
net_sum = annual.groupby("위생업태명")["net_growth"].sum().reset_index()
net_sum.columns = ["위생업태명", "5년 Net Growth"]

최종 = 폐업률표.reset_index().merge(net_sum, on="위생업태명", how="left")

print(최종.sort_values("폐업률", ascending=False))

              위생업태명  총업소수   폐업수         폐업률  5년 Net Growth
4                까페     1     1  100.000000              0
2                기타  6340  3070   48.422713           3270
11  외국음식전문점(인도,태국등)   157    68   43.312102             89
16             키즈카페     7     3   42.857143              4
8                분식   862   366   42.459397            496
21               한식  7315  2981   40.751880           4334
23               횟집   485   185   38.144330            300
13       정종/대포집/소주방   469   168   35.820896            301
19          패밀리레스트랑    32    11   34.375000             21
0              감성주점    24     8   33.333333             16
3           김밥(도시락)    48    16   33.333333             32
1               경양식   708   229   32.344633            479
22            호프/통닭  1138   364   31.985940            774
6             라이브카페    44    14   31.818182             30
9               뷔페식    83    26   31.325301             57
7              복어취급     7     2   28.571429             